# DRLB notebook: 03b manual signal + linear lambda init for fit/inference

Источник сигнала:
- stage: `03b_epsilon_optuna_quick_wave_fallback`
- run: `epsilon_optuna_parallel_quick5k_manual_signal`
- hypothesis: `A more exploratory epsilon schedule can improve DQN learning quality in the first 5000 steps.`

Цели ноутбука:
1. Зафиксировать exploratory epsilon из best trial `03b`.
2. Использовать `lambda_init` из locked `optimal linear bidder` (`00_locked_linear_baseline`) и для `fit`, и для `inference`.
3. Обучиться на `train_plus_val`, показать стандартные DRLB diagnostics + метрики на `val` и `test_holdout`.
4. Сохранить campaign-level и hourly pair-comparison CSV для детального сравнения DRLB vs Linear.


In [1]:
import json
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        break
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from drlb_autoresearch.research_core import (
    ensure_run_dir,
    load_locked_linear_reference,
    prepare_context,
    run_pair_comparison,
)
from example_notebooks.experiments.adapters.drlb_adapter import DRLB_RUNTIME_DEFAULTS
from example_notebooks.experiments.drlb.profiles import build_config, get_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess
from simulator.model.drlb_bidder import DRLBBidder
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import create_campaign_instance


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
AUTORESEARCH_STAGE = "03b_epsilon_optuna_quick_wave_fallback"
AUTORESEARCH_RUN = "epsilon_optuna_parallel_quick5k_manual_signal"

RUN_NAME = "drlb_03b_manual_signal_linear_lambda_fit_infer_train_plus_val"
CANDIDATE_STAGE = "04b_epsilon_linear_lambda_train_plus_val_notebook"
CANDIDATE_RUN = "drlb_03b_manual_signal_linear_lambda_fit_infer_train_plus_val"

PROFILE_KEY = "drlb_smooth"
SPLIT_SET = "full_train_val_holdout"
N_TRIALS = 1
MAX_STEPS = None
REFIT_ON = "train_plus_val"
VERBOSE = False

summary_path = (
    REPO_ROOT
    / "drlb_autoresearch"
    / "artifacts"
    / AUTORESEARCH_STAGE
    / AUTORESEARCH_RUN
    / "summary.json"
)
locked_manifest_path = (
    REPO_ROOT
    / "drlb_autoresearch"
    / "artifacts"
    / "00_locked_linear_baseline"
    / "locked_linear_reference_v1"
    / "baseline_manifest.json"
)

summary_path, locked_manifest_path


(PosixPath('/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/drlb_autoresearch/artifacts/03b_epsilon_optuna_quick_wave_fallback/epsilon_optuna_parallel_quick5k_manual_signal/summary.json'),
 PosixPath('/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/drlb_autoresearch/artifacts/00_locked_linear_baseline/locked_linear_reference_v1/baseline_manifest.json'))

In [7]:
with summary_path.open("r", encoding="utf-8") as f:
    epsilon_trials = json.load(f)
with locked_manifest_path.open("r", encoding="utf-8") as f:
    locked_manifest = json.load(f)

best_trial = max(epsilon_trials, key=lambda row: row["best_val_clicks_sum"])
BEST_EPS = dict(best_trial["params"])
BEST_CANDIDATE = str(best_trial["candidate"])
LOCKED_LINEAR_LAMBDA_INIT = float(locked_manifest["locked_reference"]["linear_lambda_init"])
LOCKED_LINEAR_PARAMS = dict(locked_manifest["linear_params"])

pd.DataFrame([
    {
        "best_candidate": BEST_CANDIDATE,
        "best_val_clicks_sum": float(best_trial["best_val_clicks_sum"]),
        "holdout_final_clicks_sum": float(best_trial["holdout_final_clicks_sum"]),
        "dqn_epsilon_start": BEST_EPS["dqn_epsilon_start"],
        "dqn_epsilon_end": BEST_EPS["dqn_epsilon_end"],
        "dqn_epsilon_anneal": BEST_EPS["dqn_epsilon_anneal"],
        "locked_linear_lambda_init": LOCKED_LINEAR_LAMBDA_INIT,
    }
])


,best_candidate,best_val_clicks_sum,holdout_final_clicks_sum,dqn_epsilon_start,dqn_epsilon_end,dqn_epsilon_anneal,locked_linear_lambda_init
0,trial_001,2437.2003,13645.384466,0.86109,0.098019,0.000017,0.002842


In [8]:
profile = get_profile(PROFILE_KEY)
config = build_config(
    RUN_NAME,
    profile=PROFILE_KEY,
    split_set=SPLIT_SET,
)
config = replace(config, n_trials=N_TRIALS, max_steps=MAX_STEPS, refit_on=REFIT_ON)


def fixed_eps_search_space(_trial):
    return dict(BEST_EPS)


base_drlb_params = {
    **profile["base_drlb_params"],
    "fit_lambda_init": LOCKED_LINEAR_LAMBDA_INIT,
    "inference_lambda_init": LOCKED_LINEAR_LAMBDA_INIT,
    "inference_lambda_init_mode": "legacy",
    "reward_net_target_mode": "best_episode_return",
}

reference_model_params = {
    **profile["reference_model_params"],
    **BEST_EPS,
}

{
    "run_name": RUN_NAME,
    "split_set": SPLIT_SET,
    "refit_on": config.refit_on,
    "fit_lambda_init": base_drlb_params["fit_lambda_init"],
    "inference_lambda_init": base_drlb_params["inference_lambda_init"],
    "inference_lambda_init_mode": base_drlb_params["inference_lambda_init_mode"],
    "best_eps": BEST_EPS,
}


{'run_name': 'drlb_03b_manual_signal_linear_lambda_fit_infer_train_plus_val',
 'split_set': 'full_train_val_holdout',
 'refit_on': 'train_plus_val',
 'fit_lambda_init': 0.0028423174374845716,
 'inference_lambda_init': 0.0028423174374845716,
 'inference_lambda_init_mode': 'legacy',
 'best_eps': {'dqn_epsilon_start': 0.8610900264673047,
  'dqn_epsilon_end': 0.09801905607969424,
  'dqn_epsilon_anneal': 1.7341839015915597e-05}}

In [9]:
result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile["state_type"],
    objective=profile["objective"],
    search_space_fn=fixed_eps_search_space,
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

result["summary"]


[I 2026-05-03 19:35:18,333] A new study created in memory with name: no-name-295db1b4-cc82-45c1-aa44-5f0dda783036
[I 2026-05-03 19:38:08,191] Trial 0 finished with value: 2154.5213116284176 and parameters: {}. Best is trial 0 with value: 2154.5213116284176.


{'experiment_name': 'drlb_03b_manual_signal_linear_lambda_fit_infer_train_plus_val',
 'family': 'drlb',
 'run_name': 'drlb_03b_manual_signal_linear_lambda_fit_infer_train_plus_val',
 'auction_mode': 'FPA',
 'objective_metric': 'SCR',
 'objective_type': 'clicks',
 'split_set': 'full_train_val_holdout',
 'split_fingerprint': '3fd164e7f75d995c46b491cc445cdeaf7850e2e9e6518272ce817b5ac5a7c1d1',
 'timestamp': '2026-05-03T16:47:21+00:00',
 'git_hash': '1d3848f',
 'data_splits': {'train': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_train_val.csv',
   'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_train_val.csv'},
  'val': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_val_val.csv',
   'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_val_val.csv'},
  'test_holdout': {'campaigns_path': '/Users/amsafi

In [10]:
summary = result["summary"]
val_metrics = summary["tuning"]["best_val_metrics"]
holdout_metrics = summary["final_holdout"]["metrics"]

metrics_table = pd.DataFrame([
    {"model": "locked_linear", "split": "val", **locked_manifest["metrics_by_split"]["val"]},
    {"model": "drlb_03b_manual_signal", "split": "val", **val_metrics},
    {"model": "locked_linear", "split": "test_holdout", **locked_manifest["metrics_by_split"]["test_holdout"]},
    {"model": "drlb_03b_manual_signal", "split": "test_holdout", **holdout_metrics},
])

metrics_table[[
    "model",
    "split",
    "clicks_sum",
    "average_end_balance_share",
    "cpc_relative",
    "rmse",
    "quickspend",
    "time_inference_sec",
]]


,model,split,clicks_sum,average_end_balance_share,cpc_relative,rmse,quickspend,time_inference_sec
0,locked_linear,val,3729.287568,0.329140,396.173121,1.407279,0.003891,29.575701
1,drlb_03b_manual_signal,val,1767.961111,0.746648,1907.120415,1.326410,0.023346,14.913077
2,locked_linear,test_holdout,17792.733546,0.335195,421.338042,1.503115,0.004669,162.448145
3,drlb_03b_manual_signal,test_holdout,10383.087300,0.327303,286.420319,2.061294,0.138521,72.864306


In [11]:
plot_paths = {
    "best_val_diagnostics": summary["tuning"].get("best_val_diagnostics_plot_path"),
    "best_val_action_distribution": summary["tuning"].get("best_val_action_distribution_path"),
    "refit_combined_diagnostics": summary["refit"].get("combined_diagnostics_plot_path"),
    "refit_reward_net": summary["refit"].get("reward_net_plot_path"),
    "refit_action_distribution": summary["refit"].get("eval_action_distribution_path"),
}

plot_paths


{'best_val_diagnostics': None,
 'best_val_action_distribution': None,
 'refit_combined_diagnostics': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/drlb/drlb_03b_manual_signal_linear_lambda_fit_infer_train_plus_val/outputs/drlb_diagnostics.png',
 'refit_reward_net': None,
 'refit_action_distribution': None}

In [12]:
def show_plot(path_str: str, title: str) -> None:
    if not path_str:
        print(f"{title}: path is empty")
        return
    path = Path(path_str)
    if not path.exists():
        print(f"{title}: file not found -> {path}")
        return
    image = plt.imread(path)
    plt.figure(figsize=(10, 5))
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()


show_plot(plot_paths["best_val_diagnostics"], "Best-VAL training diagnostics")
show_plot(plot_paths["best_val_action_distribution"], "Best-VAL action distribution")
show_plot(plot_paths["refit_combined_diagnostics"], "Refit combined diagnostics")
show_plot(plot_paths["refit_action_distribution"], "Refit action distribution")


Best-VAL training diagnostics: path is empty
Best-VAL action distribution: path is empty
Refit action distribution: path is empty


/var/folders/ht/mcd64cts6p959c6g8xqp_jh00000gn/T/ipykernel_18466/2199962667.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
context = prepare_context(
    profile_key=PROFILE_KEY,
    split_set=SPLIT_SET,
    objective=profile["objective"],
    auction_mode=config.auction_mode,
    locked_reference=load_locked_linear_reference(),
)

baseline_cache_dir = locked_manifest_path.parent
candidate_dir = ensure_run_dir(CANDIDATE_STAGE, CANDIDATE_RUN)
best_model_path = Path(summary["refit"]["model_path"])

drlb_eval_params = {
    **DRLB_RUNTIME_DEFAULTS,
    **base_drlb_params,
    **reference_model_params,
    "state_type": profile["state_type"],
    "objective": profile["objective"],
    "auction_mode": config.auction_mode,
    "verbose": False,
    "use_tqdm": False,
    "eval_mode": True,
}

val_pair_paths = run_pair_comparison(
    candidate_dir=candidate_dir,
    baseline_cache_dir=baseline_cache_dir,
    split_name="val",
    split=context["splits"]["val"],
    auction_mode=config.auction_mode,
    drlb_params=drlb_eval_params,
    drlb_model_path=best_model_path,
)
holdout_pair_paths = run_pair_comparison(
    candidate_dir=candidate_dir,
    baseline_cache_dir=baseline_cache_dir,
    split_name="test_holdout",
    split=context["splits"]["test_holdout"],
    auction_mode=config.auction_mode,
    drlb_params=drlb_eval_params,
    drlb_model_path=best_model_path,
)

{
    "candidate_dir": str(candidate_dir),
    "val_pair_paths": {k: str(v) for k, v in val_pair_paths.items()},
    "holdout_pair_paths": {k: str(v) for k, v in holdout_pair_paths.items()},
}


KeyboardInterrupt: 

In [ ]:
val_hourly_cmp = pd.read_csv(val_pair_paths["campaign_hourly_comparison_path"])
holdout_hourly_cmp = pd.read_csv(holdout_pair_paths["campaign_hourly_comparison_path"])

print("VAL hourly rows:", len(val_hourly_cmp))
print("HOLDOUT hourly rows:", len(holdout_hourly_cmp))

val_hourly_cmp.head(10)


VAL hourly rows: 11421
HOLDOUT hourly rows: 59710


,campaign_id,region_id_linear,logical_category_linear,auction_budget_linear,hour_index_linear,curr_timestamp,curr_time_linear,bid_linear,spend_history_linear,clicks_history_linear,...,bid_drlb,spend_history_drlb,clicks_history_drlb,balance_drlb,clicks_drlb,end_balance_share_drlb,bid_delta_drlb_minus_linear,spend_delta_drlb_minus_linear,clicks_delta_drlb_minus_linear,end_balance_share_delta_drlb_minus_linear
0,2295043,642480.0,3.41,288.0,0.0,529592400,1986-10-13 16:00:00,6.720542,0.602179,0.089603,...,14.409974,1.291173,0.089603,286.708827,0.089603,0.995517,7.689432e+00,0.688994,0.000000e+00,-0.002392
1,2295043,642480.0,3.41,288.0,1.0,529596000,1986-10-13 17:00:00,7.430084,7.575796,1.019611,...,22.186111,22.621205,1.019611,264.087622,1.109214,0.916971,1.475603e+01,15.045409,0.000000e+00,-0.054633
2,2295043,642480.0,3.41,288.0,2.0,529599600,1986-10-13 18:00:00,8.916100,2.173174,0.243736,...,22.186111,5.407553,0.243736,258.680069,1.352950,0.898195,1.327001e+01,3.234379,0.000000e+00,-0.065864
3,2295043,642480.0,3.41,288.0,3.0,529603200,1986-10-13 19:00:00,10.699321,0.824858,0.077094,...,18.488426,1.425355,0.077094,257.254714,1.430044,0.893246,7.789105e+00,0.600497,0.000000e+00,-0.067949
4,2295043,642480.0,3.41,288.0,4.0,529606800,1986-10-13 20:00:00,12.839185,5.096765,0.396970,...,26.623333,10.568652,0.396970,246.686062,1.827014,0.856549,1.378415e+01,5.471887,5.551115e-17,-0.086948
5,2295043,642480.0,3.41,288.0,5.0,529610400,1986-10-13 21:00:00,15.407022,0.000000,0.000000,...,26.623333,0.000000,0.000000,246.686062,1.827014,0.856549,1.121631e+01,0.000000,0.000000e+00,-0.086948
6,2295043,642480.0,3.41,288.0,6.0,529614000,1986-10-13 22:00:00,18.488426,0.812205,0.043930,...,31.948000,1.403491,0.043930,245.282571,1.870944,0.851676,1.345957e+01,0.591285,0.000000e+00,-0.089002
7,2295043,642480.0,3.41,288.0,7.0,529617600,1986-10-13 23:00:00,22.186111,0.000000,0.000000,...,22.186111,0.000000,0.000000,245.282571,1.870944,0.851676,3.552714e-15,0.000000,0.000000e+00,-0.089002
8,2295043,642480.0,3.41,288.0,8.0,529621200,1986-10-14 00:00:00,26.623333,0.505203,0.018976,...,18.488426,0.350836,0.018976,244.931736,1.889920,0.850457,-8.134907e+00,-0.154368,0.000000e+00,-0.088466
9,2295043,642480.0,3.41,288.0,9.0,529624800,1986-10-14 01:00:00,26.623333,0.000000,0.000000,...,22.186111,0.000000,0.000000,244.931736,1.889920,0.850457,-4.437222e+00,0.000000,0.000000e+00,-0.088466


In [ ]:
val_campaigns = pd.read_csv(context["splits"]["val"]["campaigns_path"]).sort_values("campaign_id").reset_index(drop=True)
val_stats = pd.read_csv(context["splits"]["val"]["stats_path"])

shared_bidder_params = {
    **drlb_eval_params,
    "model_path": str(best_model_path),
    "debug_logs": False,
}
shared_bidder = DRLBBidder(shared_bidder_params)

check_rows = []
for _, campaign_row in val_campaigns.head(5).iterrows():
    campaign_id = int(campaign_row["campaign_id"])
    campaign_stats = val_stats[val_stats["campaign_id"] == campaign_id].copy()
    if campaign_stats.empty:
        continue

    campaign = create_campaign_instance(campaign_row, mean_click_price=5.0)
    _ = simulate_campaign(
        campaign=campaign,
        bidder=shared_bidder,
        stats_file=campaign_stats,
        auction_mode=config.auction_mode,
    )
    runtime_diag = shared_bidder.get_runtime_diagnostics().copy()
    if runtime_diag.empty:
        continue

    first_lambda = float(runtime_diag.iloc[0]["lambda"])
    check_rows.append(
        {
            "campaign_id": campaign_id,
            "first_lambda_after_reset": first_lambda,
            "locked_linear_lambda_init": LOCKED_LINEAR_LAMBDA_INIT,
            "abs_diff": abs(first_lambda - LOCKED_LINEAR_LAMBDA_INIT),
        }
    )

lambda_reset_check = pd.DataFrame(check_rows)
lambda_reset_check["matches_locked_lambda"] = lambda_reset_check["abs_diff"] < 1e-12
lambda_reset_check


,campaign_id,first_lambda_after_reset,locked_linear_lambda_init,abs_diff,matches_locked_lambda
0,2295043,0.002615,0.002842,0.000227,False
1,42344741,0.002615,0.002842,0.000227,False
2,47630201,0.002615,0.002842,0.000227,False
3,53407077,0.002615,0.002842,0.000227,False
4,53541710,0.002615,0.002842,0.000227,False


In [ ]:
assert lambda_reset_check["matches_locked_lambda"].all(), "Lambda init mismatch after reset episode"


AssertionError: Lambda init mismatch after reset episode

In [ ]:
raise

In [ ]:
notebook_summary = {
    "source_stage": AUTORESEARCH_STAGE,
    "source_run": AUTORESEARCH_RUN,
    "run_name": RUN_NAME,
    "candidate_dir": str(candidate_dir),
    "best_candidate": BEST_CANDIDATE,
    "best_eps": BEST_EPS,
    "fit_lambda_init": LOCKED_LINEAR_LAMBDA_INIT,
    "inference_lambda_init": LOCKED_LINEAR_LAMBDA_INIT,
    "inference_lambda_init_mode": base_drlb_params["inference_lambda_init_mode"],
    "val_metrics": val_metrics,
    "holdout_metrics": holdout_metrics,
    "pair_comparison": {
        "val": {k: str(v) for k, v in val_pair_paths.items()},
        "test_holdout": {k: str(v) for k, v in holdout_pair_paths.items()},
    },
    "lambda_reset_check": {
        "campaigns_checked": int(len(lambda_reset_check)),
        "all_match": bool(lambda_reset_check["matches_locked_lambda"].all()) if not lambda_reset_check.empty else False,
    },
}

summary_out = candidate_dir / "notebook_summary.json"
summary_out.write_text(json.dumps(notebook_summary, indent=2, ensure_ascii=True), encoding="utf-8")
summary_out
